# Semana 6 — Soluciones M2: Narrow Transformations
**Bootcamp:** Fundamentos de Ingeniería de Datos — Databricks SQL

**Instructor:** Luciano Argolo | [lucianoargolo.com](https://lucianoargolo.com)

**Scope:** spark.table, select/drop/rename, filter variado, transformaciones vs acciones, anti-patrón, withColumn+when, pipeline completo, spark.sql() comparación, spark.sql()+f-strings.

---
## Parte 1: Operaciones Narrow básicas
---

### Ejercicio 2.1: select, drop, rename — SETUP

**Objetivo:** Practicar las 3 formas de manipular columnas (narrow transformations).

In [0]:
from pyspark.sql.functions import col

# Leer Silver y verificar schema
df_silver = spark.table("bootcamp.silver.propiedades")
df_silver.printSchema()
print(f"Total registros: {df_silver.count()}")

In [0]:
# .select() — elegir columnas
df_silver.select("partido", "precio", "moneda").show(5)

# .select() con expresión calculada
df_silver.select(
    col("partido"), 
    col("precio"),
    (col("precio") / 1000).alias("precio_miles")
).show(5)

# .drop() — sacar columnas
df_sin_url = df_silver.drop("url", "fecha_publicacion")
print(f"Columnas originales: {len(df_silver.columns)} → Sin url/fecha: {len(df_sin_url.columns)}")

# .withColumnRenamed()
df_silver.withColumnRenamed("metros_cuadrados_totales", "m2_totales").select("partido", "m2_totales", "precio").show(5)

### Ejercicio 2.2: filter variado — GUIDED


In [0]:
# Igualdad
df_silver.filter(col("ambientes") == 3).select("partido", "ambientes", "precio").show(5)

# Rango con .between()
df_silver.filter(col("precio").between(100000, 200000)).select("partido", "precio").show(5)

# Texto con .contains()
df_silver.filter(col("partido").contains("Capital")).select("partido", "precio").show(5)

# Lista con .isin()
df_silver.filter(col("partido").isin("Capital Federal", "Vicente López", "San Isidro")).select("partido", "precio").show(5)

# Negación con ~
df_silver.filter(~col("moneda").isin("USD")).select("partido", "moneda", "precio").show(5)

---
## Parte 2: Transformaciones, acciones y pipelines
---

### Ejercicio 2.3: Pipeline con explain() — GUIDED

In [0]:
# Pipeline de narrow transformations — no se ejecuta nada
df_venta_usd = (
    df_silver
    .filter((col("tipo_operacion") == "Venta") & (col("moneda") == "USD"))
    .filter(col("metros_cuadrados_totales") > 100)
    .select("partido", "precio", "metros_cuadrados_totales", "precio_por_m2", "ambientes")
)

print("Transformaciones aplicadas. Spark no ejecutó nada todavía.")

In [0]:
# .explain(True) — Catalyst fusionó filtros + column pruning
df_venta_usd.explain(True)

In [0]:
# .show() — acción que dispara el DAG
df_venta_usd.show(10)

### Ejercicio 2.4: Eficiente vs anti-patrón — GUIDED


In [0]:
# (A) Pipeline eficiente: 1 sola acción al final
df_eficiente = (
    df_silver
    .filter(col("tipo_operacion") == "Venta")
    .filter(col("moneda") == "USD")
    .select("partido", "precio", "precio_por_m2")
    .filter(col("precio") > 100000)
)
df_eficiente.show(5)  # 1 sola ejecución del DAG

In [0]:
# (B) Anti-patrón: acciones intermedias (3 ejecuciones separadas del DAG)
df_filtrado = df_silver.filter(col("tipo_operacion") == "Venta").select("partido", "precio", "precio_por_m2")
print(f"Registros filtrados: {df_filtrado.count()}")  # ⚠️ ACCIÓN 1

df_caro = df_filtrado.filter(col("precio") > 100000)
print(f"Propiedades caras: {df_caro.count()}")  # ⚠️ ACCIÓN 2

df_caro.show(5)  # ⚠️ ACCIÓN 3

In [0]:
# Resumen: qué es transformación (lazy) y qué es acción (ejecuta)
print("TRANSFORMACIONES (lazy — arma el plan pero no ejecuta):")
print("  .filter()   .select()    .withColumn()     .drop()")
print("  .orderBy()  .groupBy()   .join()           .distinct()")
print("  .union()    .withColumnRenamed()            .limit()")
print()
print("ACCIONES (ejecutan el DAG completo hasta ese punto):")
print("  .show()     .count()     .collect()        .first()")
print("  .take(n)    .toPandas()  .write.save...()  .foreach()")
print()
print("REGLA: 1 sola acción al final del pipeline = máxima optimización")

In [0]:
from pyspark.sql.functions import when

# withColumn + when = CASE WHEN de SQL
df_categorizado = df_venta_usd.withColumn(
    "categoria_precio",
    when(col("precio") < 80000, "Económica")
    .when(col("precio") < 200000, "Media")
    .when(col("precio") < 500000, "Premium")
    .otherwise("Lujo")
)
df_categorizado.show(15)

---
## Parte 3: Pipeline completo y spark.sql()
---

### Ejercicio 2.6: Pipeline completo — INDEPENDENT

In [0]:
# Pipeline eficiente: 1 sola acción al final
resultado_alquiler = (
    spark.table("bootcamp.silver.propiedades")
    .filter((col("tipo_operacion") == "Alquiler") & (col("moneda") == "ARS"))
    .select("partido", "precio", "ambientes", "metros_cuadrados_totales")
    .filter(col("ambientes") > 0)
    .withColumn("precio_por_ambiente", (col("precio") / col("ambientes")).cast("decimal(10,2)"))
    .orderBy(col("precio_por_ambiente").desc())
)
resultado_alquiler.show(20)

### Ejercicio 2.7: spark.sql() vs DataFrame API — INDEPENDENT

**Objetivo:** Comprobar que SQL y DataFrame API generan el mismo plan físico.

In [0]:
# Misma query en spark.sql()
resultado_sql = spark.sql("""
    SELECT 
        partido,
        precio,
        ambientes,
        metros_cuadrados_totales,
        CAST(precio / ambientes AS DECIMAL(10,2)) AS precio_por_ambiente
    FROM bootcamp.silver.propiedades
    WHERE tipo_operacion = 'Alquiler' 
      AND moneda = 'ARS'
      AND ambientes > 0
    ORDER BY precio_por_ambiente DESC
""")
resultado_sql.show(20)

In [0]:
# .explain() en ambas versiones — mismo plan
print("=== DataFrame API ===")
resultado_alquiler.explain()

print("\n=== spark.sql() ===")
resultado_sql.explain()

### Ejercicio 2.8: spark.sql() + f-strings — INDEPENDENT

**Objetivo:** Usar `spark.sql()` con f-strings para parametrizar queries y recorrer catálogos — lo que `%sql` no puede hacer.


In [0]:
from pyspark.sql.functions import avg, count, round as spark_round

# (1) Loop por zonas con query parametrizada
zonas_interes = ["Capital Federal", "Vicente López", "San Isidro", "Tigre"]

print(f"{'Zona':<20} {'Cantidad':>10} {'Precio Prom USD':>18}")
print("-" * 50)
for zona in zonas_interes:
    resultado = spark.sql(f"""
        SELECT COUNT(*) as cantidad, ROUND(AVG(precio), 2) as precio_promedio
        FROM bootcamp.silver.propiedades
        WHERE partido = '{zona}' AND tipo_operacion = 'Venta' AND moneda = 'USD'
    """).collect()[0]
    print(f"{zona:<20} {resultado['cantidad']:>10,} {resultado['precio_promedio']:>18,.2f}")

In [0]:
# (2) Recorrer catálogo completo: registros + columnas por tabla
tablas = [
    "bootcamp.bronze.properties_bronze",
    "bootcamp.silver.propiedades",
    "bootcamp.gold.fact_propiedades",
    "bootcamp.gold.dim_zona",
    "bootcamp.gold.dim_caracteristicas",
    "bootcamp.gold.dim_tipo_operacion",
    "bootcamp.gold.dim_orientacion",
    "bootcamp.gold.dim_tiempo",
]

print(f"{'Tabla':>45} {'Registros':>12} {'Columnas':>10}")
print("-" * 69)
for tabla in tablas:
    df = spark.table(tabla)
    print(f"{tabla:>45} {df.count():>12,} {len(df.columns):>10}")

In [0]:
# (3) DESCRIBE dinámico de cada tabla Gold
tablas_gold = ["fact_propiedades", "dim_zona", "dim_caracteristicas", "dim_tipo_operacion", "dim_orientacion", "dim_tiempo"]

for t in tablas_gold:
    print(f"\n{'='*60}")
    print(f"  bootcamp.gold.{t}")
    print(f"{'='*60}")
    spark.sql(f"DESCRIBE bootcamp.gold.{t}").show(truncate=False)

---
### Reflexión M2

Respondé con tus palabras:

1. **¿Por qué es un anti-patrón poner `.count()` en medio de un pipeline?** ¿Cuántas veces se ejecuta el DAG si tenés 3 acciones?
2. **Si tenés una query compleja en SQL y necesitás pasarla a PySpark, ¿qué estrategia usarías?** ¿Empezarías de cero o usarías `spark.sql()` como paso intermedio?